In [3]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.append(root_path)

In [4]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=False)

In [ ]:
from hummingbot.strategy_v2.utils.distributions import Distributions
from controllers.market_making.pmm_simple import PMMSimpleConfig
from controllers.market_making.pz_mm import PZMMControllerConfig
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
import datetime
from decimal import Decimal

# Controller configuration
connector_name = "binance"
trading_pair = "WLD-USDT"
total_amount_quote = 1000
take_profit = 0.003
stop_loss = 0.003
trailing_stop_activation_price = 0.001
trailing_stop_trailing_delta = 0.0005
time_limit = 60 * 5
executor_refresh_time = 60
cooldown_time = 60
tp_natr_factor = 0.5
sl_natr_factor = 3

# Backtesting configuration
start = int(datetime.datetime(2025, 3, 30).timestamp())
end = int(datetime.datetime(2025, 3, 31).timestamp())
# start = int(datetime.datetime(2024, 8, 1).timestamp())
# end = int(datetime.datetime(2024, 8, 2).timestamp())
backtesting_resolution = "1m"

config = PZMMControllerConfig(
    connector_name=connector_name,
    trading_pair=trading_pair,
    sell_spreads=Distributions.arithmetic(3, 0.002, 0.001),
    buy_spreads=Distributions.arithmetic(3, 0.002, 0.001),
    total_amount_quote=Decimal(total_amount_quote),
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(activation_price=Decimal(trailing_stop_activation_price), trailing_delta=Decimal(trailing_stop_trailing_delta)),
    time_limit=time_limit,
    cooldown_time=cooldown_time,
    executor_refresh_time=executor_refresh_time,
    hma_very_slow = 50,
    hma_slow = 20,
    hma_fast = 10,
    rsi_length = 9,
    stoch_rsi_smoothing = 3,
    stoch_rsi_length =9,
    natr_length = 14,
    tp_natr_factor = 0.5,
    sl_natr_factor = 3
)

In [6]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results
trade_cost = 0.001
backtesting_result = await backtesting.run_backtesting(config, start, end, backtesting_resolution, trade_cost=trade_cost)

2025-04-03 17:30:23,830 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x70b1ba259ba0>
2025-04-03 17:30:23,831 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x70b1ba216bc0>, 16466.736978247)])']
connector: <aiohttp.connector.TCPConnector object at 0x70b1ba259960>
2025-04-03 17:30:25,190 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x70b1b08bcd90>
2025-04-03 17:30:25,192 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x70b1ba215ea0>, 16468.096964722)])']
connector: <aiohttp.connector.TCPConnector object at 0x70b1b08bcdc0>
2025-04-03 17:30:25,481 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x70b1ba22ba30>
2025-04-03 17:30:25,482 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.cli

In [11]:
backtesting_result.controller_config.dict()

{'id': '6sztbmweLyWYQQGLFKJmeP7uSGB5CLotT1p7RvYLpgLU',
 'controller_name': 'pz_mm',
 'controller_type': 'market_making',
 'total_amount_quote': Decimal('1000'),
 'manual_kill_switch': None,
 'candles_config': [{'connector': 'binance',
   'trading_pair': 'WLD-USDT',
   'interval': '5m',
   'max_records': 150}],
 'connector_name': 'binance',
 'trading_pair': 'WLD-USDT',
 'buy_spreads': [0.002, 0.003, 0.004],
 'sell_spreads': [0.002, 0.003, 0.004],
 'buy_amounts_pct': [Decimal('1'), Decimal('1'), Decimal('1')],
 'sell_amounts_pct': [Decimal('1'), Decimal('1'), Decimal('1')],
 'executor_refresh_time': 60,
 'cooldown_time': 60,
 'leverage': 20,
 'position_mode': 'HEDGE',
 'stop_loss': Decimal('0.003000000000000000062450045135165055398829281330108642578125'),
 'take_profit': Decimal('0.003000000000000000062450045135165055398829281330108642578125'),
 'time_limit': 300,
 'take_profit_order_type': <OrderType.LIMIT: 2>,
 'trailing_stop': {'activation_price': Decimal('0.00100000000000000002081668

In [12]:
# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
backtesting_result.get_backtesting_figure()


Net PNL: $-57.42 (-5.74%) | Max Drawdown: $-63.80 (-6.38%)
Total Volume ($): 447089.56 | Sharpe Ratio: -2.00 | Profit Factor: 0.74
Total Executors: 1926 | Accuracy Long: 0.63 | Accuracy Short: 0.49
Close Types: Take Profit: 96 | Stop Loss: 150 | Time Limit: 792 |
             Trailing Stop: 303 | Early Stop: 585



In [13]:
# 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df
executors_df.head()

,id,timestamp,type,close_timestamp,close_type,status,config,net_pnl_pct,net_pnl_quote,cum_fees_quote,filled_amount_quote,is_active,is_trading,custom_info,controller_id,side
0,EywoEdoK3JdpVzWhiWtRXuXjegv4AtuAStUKNnzLamxx,1743293100,position_executor,1743293280,CloseType.TRAILING_STOP,RunnableStatus.TERMINATED,{'id': 'EywoEdoK3JdpVzWhiWtRXuXjegv4AtuAStUKNn...,0.00029032258064520897328231185952063242439180...,0.04841258144033216276325148896830796729773283...,0.16675444718333895521844567610969534143805503...,333.50889436667790732826688326895236968994140625,False,False,"{'close_price': 0.774, 'level_id': 'sell_0', '...",None,SELL
1,9NXR2iyVfUxJefJLNzVxeYgJdwtHF8uKmM15CzWnMCfJ,1743293100,position_executor,1743293280,CloseType.TRAILING_STOP,RunnableStatus.TERMINATED,{'id': '9NXR2iyVfUxJefJLNzVxeYgJdwtHF8uKmM15Cz...,0.00029032258064520897328231185952063242439180...,0.04841240364918789756965011861211678478866815...,0.16675383479161981492566724227799568325281143...,333.50766958323964672672445885837078094482421875,False,False,"{'close_price': 0.774, 'level_id': 'sell_1', '...",None,SELL
2,uxVEqvEz7FUDuWXf4jD2CSijtqcAA2mVcYWs2L981uG,1743293100,position_executor,1743293280,CloseType.TRAILING_STOP,RunnableStatus.TERMINATED,{'id': 'uxVEqvEz7FUDuWXf4jD2CSijtqcAA2mVcYWs2L...,0.00029032258064520897328231185952063242439180...,0.04841222585934946975871895347154350019991397...,0.16675322240439857668370393639634130522608757...,333.506444808797141376999206840991973876953125,False,False,"{'close_price': 0.774, 'level_id': 'sell_2', '...",None,SELL
3,4zU6stxvJyFsmsb7REfx5BFhGasTmhF6L1Kn5XhvfvD2,1743293100,position_executor,1743293400,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': '4zU6stxvJyFsmsb7REfx5BFhGasTmhF6L1Kn5X...,-0.0010000000000001110431191442273757274961099...,-0.1665417266058007994544709617912303656339645...,0.16654172660578231424111095293483231216669082...,333.08345321156463114675716497004032135009765625,False,False,"{'close_price': 0.774, 'level_id': 'buy_0', 's...",None,BUY
4,8EUFJBbN9B9GksaVqCKwKoXefcEDZ4Dqk6NhLSguHHuj,1743293100,position_executor,1743293400,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': '8EUFJBbN9B9GksaVqCKwKoXefcEDZ4Dqk6NhLS...,-0.0010000000000001110431191442273757274961099...,-0.1665423382297983034838750882045133039355278...,0.16654233822977981827051507934811525046825408...,333.08467645955960279025021009147167205810546875,False,False,"{'close_price': 0.774, 'level_id': 'buy_1', 's...",None,BUY


### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [14]:
import plotly.express as px

# Create a new column for profitability
executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# Create the scatter plot
fig = px.scatter(
    executors_df,
    x="timestamp",
    y='net_pnl_quote',
    title='PNL per Trade',
    color='profitable',
    color_discrete_map={True: 'green', False: 'red'},
    labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
    hover_data=['filled_amount_quote', 'side']
)

# Customize the layout
fig.update_layout(
    xaxis_title="Timestamp",
    yaxis_title="Net PNL (Quote)",
    legend_title="Profitable",
    font=dict(size=12, color="white"),
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
    paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
    xaxis=dict(gridcolor="gray"),
    yaxis=dict(gridcolor="gray")
)

# Add a horizontal line at y=0 to clearly separate profits and losses
fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# Show the plot
fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [ ]:
fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
fig.show()
